# Ndizi LoRA error analysis

Compare **baseline Sunbird-51** vs **Ndizi LoRA** on held-out test, then inspect worst/best clips with in-notebook audio + waveforms.

Assumes you already ran:

```bash
python -m ndizi_asr.cli --model sunbird-whisper-51 \
  --dataset smutuvi/ndizi-1:test smutuvi/ndizi-1-2025:test \
  --experiment-name baseline_sunbird51_ndizi_test

python -m ndizi_asr.cli --model hf:outputs/sunbird51_ndizi_lora \
  --dataset smutuvi/ndizi-1:test smutuvi/ndizi-1-2025:test \
  --experiment-name sunbird51_ndizi_lora_test
```

On a headless GPU box, use this notebook (Jupyter / VS Code / Cursor) so audio plays in the **browser**, not via `paplay`.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import Audio, load_dataset
from dotenv import load_dotenv
from IPython.display import Audio as IPyAudio
from IPython.display import display, Markdown
from jiwer import cer as jiwer_cer
from jiwer import wer as jiwer_wer

load_dotenv()
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN", "").strip()))

# Repo root = parent of notebooks/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
print("ROOT:", ROOT.resolve())

BASELINE_DIR = ROOT / "results" / "baseline_sunbird51_ndizi_test"
LORA_DIR = ROOT / "results" / "sunbird51_ndizi_lora_test"
HUB_SPLIT = "test"
TOP_N = 5

In [ ]:
def find_metrics(exp_dir: Path) -> Path:
    hits = sorted(exp_dir.glob("*/metrics.json"))
    if hits:
        return hits[0]
    summary = exp_dir / "summary.json"
    if summary.is_file():
        return summary
    raise FileNotFoundError(f"No metrics under {exp_dir}")


def find_predictions(exp_dir: Path) -> Path:
    hits = sorted(exp_dir.glob("*/predictions.jsonl"))
    if not hits:
        raise FileNotFoundError(f"No predictions.jsonl under {exp_dir}")
    return hits[0]


def load_metrics(path: Path) -> dict:
    raw = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(raw, dict) and "overall" in raw:
        return raw
    if isinstance(raw, dict) and "models" in raw:
        return next(iter(raw["models"].values()))
    raise ValueError(f"Unrecognized metrics in {path}")


def metric(block: dict, key: str):
    return block.get(f"{key}_normalized", block.get(key))


def score_predictions(path: Path) -> pd.DataFrame:
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            if r.get("error"):
                continue
            ref_n = str(r.get("reference_normalized") or r.get("reference") or "")
            hyp_n = str(r.get("prediction_normalized") or r.get("prediction") or "")
            if not ref_n.strip():
                continue
            try:
                w = float(jiwer_wer(ref_n, hyp_n))
                c = float(jiwer_cer(ref_n, hyp_n))
            except Exception:
                w, c = 1.0, 1.0
            rows.append(
                {
                    "sample_id": r.get("sample_id"),
                    "dataset_id": r.get("dataset_id"),
                    "language": r.get("language"),
                    "wer": w,
                    "cer": c,
                    "reference": r.get("reference"),
                    "prediction": r.get("prediction"),
                    "reference_normalized": ref_n,
                    "prediction_normalized": hyp_n,
                    "latency_seconds": r.get("latency_seconds"),
                    "idx": int(str(r.get("sample_id", "0")).rsplit(":", 1)[-1]),
                }
            )
    return pd.DataFrame(rows)


def load_hub_audio(dataset_id: str, idx: int, split: str = HUB_SPLIT):
    ds = load_dataset(dataset_id, split=split)
    cols = list(ds.column_names)
    audio_col = next(c for c in ("audio", "audio_path", "path", "wav", "speech") if c in cols)
    if audio_col == "audio":
        ds = ds.cast_column(audio_col, Audio(sampling_rate=16000))
    item = ds[int(idx)]
    audio = item[audio_col]
    if isinstance(audio, dict) and "array" in audio:
        arr = np.asarray(audio["array"], dtype=np.float32)
        sr = int(audio.get("sampling_rate") or 16000)
        return arr, sr
    raise TypeError(f"Unsupported audio type: {type(audio)}")


def show_clip(row: pd.Series, *, title: str | None = None):
    """Load Hub audio, plot waveform + spectrogram, play in-notebook."""
    arr, sr = load_hub_audio(str(row["dataset_id"]), int(row["idx"]))
    dur = len(arr) / max(sr, 1)
    title = title or f"{row['sample_id']}  WER={row['wer']:.3f}  CER={row['cer']:.3f}"
    display(Markdown(f"### {title}"))
    display(Markdown(f"**dataset:** `{row['dataset_id']}` · **duration:** {dur:.1f}s · **sr:** {sr}"))
    display(Markdown(f"**REF:** {row['reference']}"))
    display(Markdown(f"**HYP:** {row['prediction']}"))

    t = np.arange(len(arr)) / sr
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=False)
    axes[0].plot(t, arr, lw=0.6, color="#1f4e79")
    axes[0].set_ylabel("amplitude")
    axes[0].set_title("Waveform")
    axes[0].grid(True, alpha=0.25)

    axes[1].specgram(arr, Fs=sr, NFFT=512, noverlap=256, cmap="magma")
    axes[1].set_ylabel("Hz")
    axes[1].set_xlabel("time (s)")
    axes[1].set_title("Spectrogram")
    fig.tight_layout()
    plt.show()

    display(IPyAudio(arr, rate=sr))
    return arr, sr

## 1. Per-dataset comparison

In [ ]:
base_m = load_metrics(find_metrics(BASELINE_DIR))
lora_m = load_metrics(find_metrics(LORA_DIR))

datasets = sorted(set(base_m.get("per_dataset") or {}) | set(lora_m.get("per_dataset") or {}))
rows = []
for ds in datasets + ["__overall__"]:
    if ds == "__overall__":
        b, l, label = base_m.get("overall") or {}, lora_m.get("overall") or {}, "ALL"
    else:
        b = (base_m.get("per_dataset") or {}).get(ds) or {}
        l = (lora_m.get("per_dataset") or {}).get(ds) or {}
        label = ds
    bw, lw = metric(b, "wer"), metric(l, "wer")
    bc, lc = metric(b, "cer"), metric(l, "cer")
    n = l.get("num_scored") or b.get("num_scored") or l.get("n") or b.get("n")
    rows.append(
        {
            "dataset": label,
            "n": n,
            "base_wer": bw,
            "lora_wer": lw,
            "delta_wer": (None if bw is None or lw is None else lw - bw),
            "base_cer": bc,
            "lora_cer": lc,
            "delta_cer": (None if bc is None or lc is None else lc - bc),
        }
    )

cmp_df = pd.DataFrame(rows)
display_df = cmp_df.copy()
for col in ["base_wer", "lora_wer", "delta_wer", "base_cer", "lora_cer", "delta_cer"]:
    display_df[col] = display_df[col].map(lambda x: None if x is None else f"{100*x:.2f}%")
display_df

Negative `delta_wer` = LoRA better. Expect overall ~44.65% → ~41.46% WER.

## 2. Score LoRA predictions and list best / worst

In [ ]:
lora_df = score_predictions(find_predictions(LORA_DIR))
print(f"scored {len(lora_df)} utterances")
print(lora_df.groupby("dataset_id")["wer"].agg(["count", "mean", "median"]))

best = lora_df.nsmallest(TOP_N, "wer")
worst = lora_df.nlargest(TOP_N, "wer")

display(Markdown("### Best (lowest WER)"))
display(best[["wer", "cer", "dataset_id", "sample_id", "reference", "prediction"]])

display(Markdown("### Worst (highest WER)"))
display(worst[["wer", "cer", "dataset_id", "sample_id", "reference", "prediction"]])

## 3. Deep dive — play + waveform for a worst clip

Change `CLIP_RANK` (0 = worst) to step through the list. Audio plays in the notebook UI.

In [ ]:
CLIP_RANK = 0  # 0 = highest WER
row = worst.iloc[CLIP_RANK]
arr, sr = show_clip(row, title=f"Worst #{CLIP_RANK + 1}")

## 4. Optional — same view for a best clip

In [ ]:
BEST_RANK = 0
row_best = best.iloc[BEST_RANK]
_ = show_clip(row_best, title=f"Best #{BEST_RANK + 1}")

## 5. Error-analysis scratchpad (extend later)

Ideas to grow this section:

- filter by `dataset_id` / WER bins
- compare baseline vs LoRA hyp for the same `sample_id`
- word-level alignments (`jiwer.process_words`)
- duration / speaking-rate vs WER scatter
- export a short HTML or CSV error sheet

In [ ]:
# Join baseline hyps on sample_id for side-by-side error review
base_df = score_predictions(find_predictions(BASELINE_DIR))
merged = lora_df.merge(
    base_df[["sample_id", "wer", "prediction"]].rename(
        columns={"wer": "base_wer", "prediction": "base_prediction"}
    ),
    on="sample_id",
    how="left",
)
merged["wer_delta"] = merged["wer"] - merged["base_wer"]

display(Markdown("### Where LoRA helped most (largest WER drop)"))
display(
    merged.nsmallest(TOP_N, "wer_delta")[
        ["wer_delta", "base_wer", "wer", "dataset_id", "sample_id", "reference", "base_prediction", "prediction"]
    ]
)

display(Markdown("### Where LoRA hurt most (largest WER rise)"))
display(
    merged.nlargest(TOP_N, "wer_delta")[
        ["wer_delta", "base_wer", "wer", "dataset_id", "sample_id", "reference", "base_prediction", "prediction"]
    ]
)

In [ ]:
# Duration vs WER scatter (loads audio lengths — can be slow the first time)
SAMPLE_FOR_SCATTER = 50  # raise later for a fuller plot
subset = lora_df.sample(min(SAMPLE_FOR_SCATTER, len(lora_df)), random_state=0)

durs = []
for _, r in subset.iterrows():
    try:
        a, s = load_hub_audio(str(r["dataset_id"]), int(r["idx"]))
        durs.append(len(a) / max(s, 1))
    except Exception:
        durs.append(np.nan)
subset = subset.copy()
subset["duration_s"] = durs

fig, ax = plt.subplots(figsize=(7, 4))
for ds, g in subset.dropna(subset=["duration_s"]).groupby("dataset_id"):
    ax.scatter(g["duration_s"], g["wer"], alpha=0.7, label=ds)
ax.set_xlabel("duration (s)")
ax.set_ylabel("WER")
ax.set_title("LoRA WER vs clip duration (sample)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Export selected clips for offline listening

If the notebook player is awkward, write WAVs next to short transcripts.

In [ ]:
import soundfile as sf

EXPORT_DIR = Path("/tmp/ndizi_lora_notebook_export/worst")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for i, (_, r) in enumerate(worst.iterrows(), 1):
    arr, sr = load_hub_audio(str(r["dataset_id"]), int(r["idx"]))
    stem = f"worst_{i:02d}_wer{r['wer']:.3f}_{str(r['sample_id']).replace(':', '_')}"
    wav = EXPORT_DIR / f"{stem}.wav"
    txt = EXPORT_DIR / f"{stem}.txt"
    sf.write(wav, arr, sr)
    txt.write_text(
        f"sample_id: {r['sample_id']}\n"
        f"dataset_id: {r['dataset_id']}\n"
        f"wer: {r['wer']}\n"
        f"REF: {r['reference']}\n"
        f"HYP: {r['prediction']}\n",
        encoding="utf-8",
    )
    print("wrote", wav)

print("\nscp example:")
print(f"  scp 'user@host:{EXPORT_DIR}/*' ./ndizi_worst/")